In [1]:
# ==========================================
# Author: George Ayensu, PE
# Project: PhD Working Code - Week 4
# Case Study: Critical Minerals Processing (SAG Mill) + BESS
# Date: November 2026
# ==========================================

!pip install pyomo highspy -q
import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.T = pyo.Set(initialize=range(1, 25))

# --- CRITICAL MINERALS MINING PARAMETERS ---
# A SAG Mill or Autoclave is an inflexible, continuous load
mill_load_mw = 20.0  # 20 MW continuous load (typical for a large mine)

# Battery Energy Storage System (BESS)
bess_capacity_mwh = 60.0  # 60 MWh capacity
bess_max_power_mw = 15.0  # Can charge/discharge at 15 MW max

# Tariff (Peak demand charges or TOU)
def price_rule(model, t):
    if t <= 6 or t >= 22: return 0.08 # Off-peak
    else: return 0.28 # Peak
model.price = pyo.Param(model.T, initialize=price_rule)

# Variables
model.grid_import = pyo.Var(model.T, within=pyo.NonNegativeReals) # Power from grid
model.bess_charge = pyo.Var(model.T, within=pyo.NonNegativeReals) # Power charging BESS
model.bess_discharge = pyo.Var(model.T, within=pyo.NonNegativeReals) # Power from BESS
model.bess_soc = pyo.Var(model.T, within=pyo.NonNegativeReals) # State of Charge

# Constraints
# 1. BESS Power Limits
model.limit_charge = pyo.Constraint(model.T, rule=lambda m, t: m.bess_charge[t] <= bess_max_power_mw)
model.limit_discharge = pyo.Constraint(model.T, rule=lambda m, t: m.bess_discharge[t] <= bess_max_power_mw)
model.limit_soc = pyo.Constraint(model.T, rule=lambda m, t: m.bess_soc[t] <= bess_capacity_mwh)

# 2. The Mill MUST run continuously (Inflexible Load Constraint)
def mill_balance_rule(model, t):
    # Grid Import + BESS Discharge = Mill Load + BESS Charge
    return model.grid_import[t] + model.bess_discharge[t] == mill_load_mw + model.bess_charge[t]
model.mill_balance = pyo.Constraint(model.T, rule=mill_balance_rule)

# 3. BESS State of Charge Balance
def bess_balance_rule(model, t):
    if t == 1:
        return model.bess_soc[t] == 0 + model.bess_charge[t] - model.bess_discharge[t]
    else:
        return model.bess_soc[t] == model.bess_soc[t-1] + model.bess_charge[t] - model.bess_discharge[t]
model.bess_balance = pyo.Constraint(model.T, rule=bess_balance_rule)

# Objective: Minimize Total Electricity Cost
model.cost = pyo.Objective(
    expr=sum(model.price[t] * model.grid_import[t] for t in model.T),
    sense=pyo.minimize
)

# Solve
solver = pyo.SolverFactory('appsi_highs')
solver.solve(model)

print(f"Total Daily Electricity Cost (Mining + BESS): ${model.cost.expr():,.2f}")
print("-------------------------------------------------------------------------")
print("Hour | Price | Mill Load | BESS Charge | BESS Discharge | Grid Import | BESS SOC")
for t in model.T:
    print(f"{t:4} | ${model.price[t]:4.2f} | {mill_load_mw:9.1f} | {model.bess_charge[t].value:11.1f} | {model.bess_discharge[t].value:14.1f} | {model.grid_import[t].value:11.1f} | {model.bess_soc[t].value:8.1f}")


Total Daily Electricity Cost (Mining + BESS): $86.40
-------------------------------------------------------------------------
Hour | Price | Mill Load | BESS Charge | BESS Discharge | Grid Import | BESS SOC
   1 | $0.08 |      20.0 |        15.0 |            0.0 |        35.0 |     15.0
   2 | $0.08 |      20.0 |        15.0 |            0.0 |        35.0 |     30.0
   3 | $0.08 |      20.0 |        15.0 |            0.0 |        35.0 |     45.0
   4 | $0.08 |      20.0 |        15.0 |            0.0 |        35.0 |     60.0
   5 | $0.08 |      20.0 |         0.0 |           -0.0 |        20.0 |     60.0
   6 | $0.08 |      20.0 |         0.0 |           -0.0 |        20.0 |     60.0
   7 | $0.28 |      20.0 |         0.0 |           15.0 |         5.0 |     45.0
   8 | $0.28 |      20.0 |         0.0 |           15.0 |         5.0 |     30.0
   9 | $0.28 |      20.0 |         0.0 |           15.0 |         5.0 |     15.0
  10 | $0.28 |      20.0 |         0.0 |           15.0 |      